In [11]:
import torch
from torch import nn
import math

In [12]:
vocab_size = 512
embed_size = 256
num_heads = 8
ff_size = 1024
num_blocks = 8
context_size = 256
batch_size = 64


In [13]:
class RoPE(nn.Module):
    def __init__(self, d_head, s_length_max, base_freq=1024):
        super(RoPE, self).__init__()

        self.d_head = d_head

        inv_freq = 1.0 / (base_freq ** (torch.arange(0, d_head, 2) / d_head))
        pos = torch.arange(0, s_length_max, dtype=torch.float)
        angle = torch.outer(pos, inv_freq)

        self.register_buffer("cos", torch.cos(angle), persistent=False)
        self.register_buffer("sin", torch.sin(angle), persistent=False)

    def forward(self, x):
        s = x.size(-2) # context size
        d = self.d_head

        x_ = x.view(*x.shape[:-1], d // 2, 2)
        x1 = x_[..., 0]
        x2 = x_[..., 1]

        cos = self.cos[:s].to(dtype=x.dtype, device=x.device)
        sin = self.sin[:s].to(dtype=x.dtype, device=x.device)

        y1 = x1 * cos - x2 * sin
        y2 = x1 * sin + x2 * cos

        y = torch.stack((y1, y2), dim=-1).view_as(x)

        return y


class Attention(nn.Module):
    def __init__(self):
        super(Attention, self).__init__()

        self.head_size = embed_size // num_heads
        self.rope = RoPE(self.head_size, context_size, 1024)
        self.qkv_linear = nn.Linear(embed_size, 3 * embed_size, bias=False)
        self.heads_fuser = nn.Linear(embed_size, embed_size, bias=False)

        self.register_buffer("mask", torch.triu(torch.full((context_size, context_size), -torch.inf), 1), persistent=False)


    def forward(self, x: torch.Tensor):
        x_ = self.qkv_linear(x)

        x_ = x_.view(*x_.shape[:2], 3, num_heads, self.head_size).transpose(1, 3)

        q, k, v = x_.unbind(dim=2)

        q, k = self.rope(q), self.rope(k)

        s = q @ k.transpose(-1, -2) / math.sqrt(self.head_size)

        mask = self.mask[:x.size(1), :x.size(1)].to(dtype=x.dtype, device=x.device)
        s += mask
        s = torch.softmax(s, -1)
        # DROPOUT
        a = s @ v
        a = a.transpose(1,2).reshape_as(x)

        y = self.heads_fuser(a)

        return y


class SwiGLU(nn.Module):
    def __init__(self):
        super(SwiGLU, self).__init__()

        self.in_layer = nn.Linear(embed_size, ff_size * 2, bias=False)
        self.silu = nn.SiLU()
        self.out_layer = nn.Linear(ff_size, embed_size, bias=False)

    def forward(self, x: torch.Tensor):
        ab = self.in_layer(x)
        a, b = ab.chunk(2, -1)

        y = a * self.silu(b)
        y = self.out_layer(y)

        return y


"""
                ____________
                |           x
                |           |
                |       RMSNorm
                |           |___________
                |           KQV         |
                |           |           |
                |   Multihead split     |
                |           |           |
                |       RoPE(Q,K)       |
                |           |           |   Attention
Attention Block |   Attention(K,Q,V)    |
                |           |           |
                |       Fuse Heads      |
                |           |           |
                |       x + |___________|
                |           |
                |       RMSNorm
                |           |
                |       SwiGLU FFN
                |           |
                |_______x + |
"""
class AttentionBlock(nn.Module):
    def __init__(self):
        super(AttentionBlock, self).__init__()

        self.pre_norm = nn.RMSNorm(embed_size)
        self.attention = Attention()
        self.post_norm = nn.RMSNorm(embed_size)
        self.swiglu = SwiGLU()


    def forward(self, x: torch.Tensor):
        y = self.attention(self.pre_norm(x))
        y = self.swiglu(self.post_norm(x + y))
        return x + y


class AttentionBlocksStack(nn.Module):
    def __init__(self, num_blocks):
        super(AttentionBlocksStack, self).__init__()

        self.atten_blocks = nn.ModuleList([AttentionBlock() for _ in range(num_blocks)])

    def forward(self, x: torch.Tensor):
        for atten_block in self.atten_blocks:
            x = atten_block(x)

        return x


class Rockformer(nn.Module):
    def __init__(self):
        super(Rockformer, self).__init__()

        self.embed = nn.Embedding(vocab_size, embed_size)
        self.atten_blocks = AttentionBlocksStack(num_blocks)
        self.post_norm = nn.RMSNorm(embed_size)
        self.reverse_embed = nn.Linear(embed_size, vocab_size, bias=False)

        self.reverse_embed.weight = self.embed.weight

        self.seq = nn.Sequential(
                                    self.embed,
                                    self.atten_blocks,
                                    self.post_norm,
                                    self.reverse_embed
                                )

    def forward(self, x):
        return self.seq(x)

In [14]:
torch.manual_seed(777)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

with torch.no_grad():
    x = torch.randint(0, vocab_size, (batch_size, context_size)).to(device)
    print(x.shape)
    model = Rockformer().to(device).eval()
    y = model(x)

print(y.shape)
y_ids = torch.argmax(y,dim=-1).cpu()
print(y_ids.shape)
print(y_ids)

device: cuda
torch.Size([64, 256])
torch.Size([64, 256, 512])
torch.Size([64, 256])
tensor([[103, 303,  59,  ...,  84, 472, 367],
        [197, 385, 324,  ..., 438, 177, 350],
        [119, 181, 318,  ...,  92, 319, 281],
        ...,
        [ 39, 444, 231,  ..., 378, 105,  19],
        [177, 302, 301,  ..., 188, 345, 479],
        [399, 327, 430,  ..., 133, 119, 344]])


In [15]:
import sentencepiece as sp

sp_model = sp.SentencePieceProcessor(model_file='gutenberg_poetry_tokenizer.model')

print(sp_model.decode_ids(x[0].cpu().tolist()))
print(sp_model.decode_ids(y_ids[0].tolist()))

ent ever e briimiutould lo asationE now9 heart ri fe someasted ear see am foiterowithing theyent y amgh moWtherlf shallful ag EstW had sun all Jwn But Cou then clrom chicngv hhatidar7hanam Ofþpp knft oldrouven sou tr turps Uusou ab? God ⁇ v moreineousm nowant as upon m gre them is love withown re himonagherefe herise p c?ow that my nightforeDot cendenount sherownging-- hand outis its riordk chS herut chaom upon gre their{ le who blIine uponisA LastAem That ear Buthereirebl St eyast FicVamf g hould'oldilOall D arare cha throughraire yill bhereetideite heetangop fl poZ there9 Hising night andil InQ dece ar d un{ thy forant him ifft B,ved n Ud When w p love youang chordF throughter gre Ihat so wol are can sw some like heartordless on? sm
ent ever e briimiutould lo asationE now9 heart ri fe someasted ear see am foiterowithing theyent y amgh moWtherlf shallful ag EstW had sun all Jwn But Cou then clrom chicngv hhatidar7hanam Ofþpp knft oldrouven sou tr turps Uusou ab? God ⁇ v moreineousm no